In [ ]:
import numpy as np
import plotly.express as plx
import plotly.graph_objects as go
import pandas as pd

N = 25 #Number of objects
Time = 250 #Duration of the simulation
frame_skip = 50 #Number of timesteps between frames

M = 100 #Mass Multiplier [M=1, objects will have a mass between 0 and 1]
R = 250 #Intital Postition Multiplier [R=1, objects will be randomly placed in a 1x1x1 cube]

vm = 0.015 #Velocity Multiplier [vm=1, objects will have a velocity between 0 and 1 in each direction]
rot_axis = np.array((0,0,1)) # (0,0,1) rotates around the z-axis
drag = 0

soft_len = 0.1 #Softening Length to prevent singularities in force calculations when objects get very close to each other
G = 1 #Gravitational Constant (set to 1 for simplicity, can be changed to real value if desired)

dt = 0.01 #Timestep
t = 0 #Intializing time
frames = [] #Setting the empty frame list
frame_num = 0 #Intializing frame number

#Setting array of objects:
    #col 0 stores mass
    #col 1,2,3 stores position in x,y,z directions
    #col 4,5,6 stores the theta, phi and radius values
    #col 7,8,9 stores momentum in x,y,z directions
    #col 10,11,12 stores force on the object in x,y,z directions
objects = np.zeros([N,13])

mass = objects[:,0]
position = objects[:,1:4] ; x,y,z = position[:,0], position[:,1], position[:,2]
sph_position = objects[:,4:7] ; theta, phi, radius = sph_position[:,0], sph_position[:,1], sph_position[:,2]
momentum = objects[:,7:10]
force = objects[:,10:13]

for i in range(N):

    #Setting a lower bound on mass to prevent very low mass objects from having very high velocities and escaping the system immediately
    mass[i] = M * (0.1 + 0.9*np.random.random())

    # Using spherical coordinates to distribute points in a sphere instead of a cube
    # rr is taken to the power of 1/3 to ensure a uniform distribution in the volume of the sphere instead of clustering towards the center
    # Setting ur = cos(phi) with a uniform distribution instead of the axis baised cos(0-pi), therefore: sin(phi) = sqrt(1-ur^2)
    thetar = np.random.uniform(0, 2*np.pi)
    ur = np.random.uniform(-1, 1)
    rr = R * np.random.random()**(1/3)

    position[i,0:3] = np.array((
        rr*np.cos(thetar)*np.sqrt(1-ur**2),
        rr*np.sin(thetar)*np.sqrt(1-ur**2),
        rr*ur
        ))
    
    #Setting a lower bound on mass to prevent very low mass objects from having very high velocities and escaping the system immediately
    mass[i] = M * R/2 * (0.1 + 0.9*np.random.random()) / (np.sqrt(x[i]**2 + y[i]**2 + z[i]**2) + 0.1)

    # Converting to spherical coordinates for calculating initial momentum as I want the system to have a net angular momentum
    sph_position[i,0:3] = np.array((
        np.arctan2(y[i], x[i]),
        np.arctan2(np.sqrt(x[i]**2 + y[i]**2), z[i]),
        np.sqrt(x[i]**2 + y[i]**2 + z[i]**2)
        ))
    
    # Computing initial momentum using the cross product of velocity and position times mass so that the axis of rotation can be easily changed
    L_axis = vm * rot_axis / np.linalg.norm(rot_axis) # Normalizing the rotation axis and multiplying by velocity multiplier to control the initial velocity of the objects
    vel_vec = np.cross(L_axis, position[i,0:3])
    momentum[i,0:3] = mass[i] * vel_vec

#Main simulation loop
while t<Time:
    #Sets force equal to 0
    force[:,0:3] = 0
    
    #Records current object position and frame
    if frame_num % frame_skip == 0:
        for i in range(N):
            frames.append((x[i], y[i], z[i], str(i), int(frame_num/frame_skip)))
    
    #Calculating the force between every object by vectorizing the calculation instead of using nested for loops to improve performance
    r = position[:, np.newaxis, 0:3] - position[np.newaxis, :, 0:3]
    r_dot = np.sum(r**2, axis=2) + soft_len**2
    dist = r_dot**(-1.5)
    np.fill_diagonal(dist, 0) #Fills diagonal with 0 to prevent singularity when i=j
    F = -G * (
    mass[:,None,None]
    * mass[None,:,None]
    * r
    * dist[:,:,None]
    )
    force[:,0:3] = np.sum(F, axis=1)
    
    #Updating momentum
    momentum[:,0:3] += force[:,0:3] * dt
    momentum *= (1 - drag*dt)
    
    #Updating Position
    for i in range(N): #Put in for loop so that mass is a scalar instead of an array of different shape from momentum. [find better solution?]
        position[i,0:3] += (momentum[i,0:3] / mass[i]) * dt
    
    t += dt
    frame_num += 1

#Converting saved position data into a format that px.scatter_3d can use
df = pd.DataFrame(frames, columns=['x','y','z','object','frame'])

fig = plx.scatter_3d(
    df,
    x='x',
    y='y',
    z='z',
    color='object',
    animation_frame='frame',
    width=800, 
    height=800,
)

fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 10
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 1

fig.update_layout(
    scene=dict(
        xaxis=dict(range=[-10*R, 10*R]),
        yaxis=dict(range=[-10*R, 10*R]),
        zaxis=dict(range=[-10*R, 10*R]),
        aspectmode='cube'
    )
)

fig.show()